# 1. Simulate: conditioning and the two openCARP runs

* **preflight** records software versions and the repository commit, and checks that the chosen
  solver flavour (boomeramg) works on this openCARP build (else petsc).
* **conditioning** paces one Courtemanche cell 1,000 times with `bench` (limit-cycle state).
* **simulation** runs `efov/opencarp/slab_experiment.py` (carputils) for the intact and lesion slab:
  monodomain prepacing to the restart time, then pseudo-bidomain with the bath.

On the workstation the full-size runs take hours. Running them from the terminal is safer than a
notebook kernel: `nohup python -m efov run --out runs/v3_reference --np 24 --until simulation > runs/sim.log 2>&1 &`.
This notebook then simply reuses the finished runs.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "final_model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v3_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
run.runStage("preflight")

In [ ]:
run.runStage("conditioning")

In [ ]:
result = run.runStage("simulation")
pd.DataFrame(result["timing"]).T.join(pd.DataFrame(result["nodes"]).T)

Each run folder keeps the exact openCARP commands (`runRecord.json`), the spec (`<condition>.json`) and the full log.

In [ ]:
record = json.loads((run.jobs["lesion"] / "runRecord.json").read_text())
print(" ".join(record["commands"]["run"])[:3000])